In [0]:
CREATE OR REPLACE TABLE data_governance.gold_cost.dim_sku_pricing 
USING DELTA
AS
SELECT
    sku_name,
    cloud,
    usage_unit,
    effective_price,
    currency_code,
    region,
    service_name,
    CURRENT_TIMESTAMP() AS load_timestamp

FROM data_governance.silver_cost_monitoring.billing_pricing

In [0]:
CREATE OR REPLACE TABLE data_governance.gold_cost.dim_cluster_nodes 
USING DELTA
AS
SELECT
    node_type,
    core_count,
    memory_gb,
    gpu_count,
    is_gpu_node,
    CURRENT_TIMESTAMP() AS load_timestamp

FROM data_governance.silver_cost_monitoring.cluster_node

In [0]:
-- STEP 1: Get last processed timestamp
WITH last_run AS (
    SELECT COALESCE(MAX(load_timestamp), TIMESTAMP('1900-01-01')) AS last_ts
    FROM data_governance.gold_cost.fact_cost_usage
),

-- STEP 2: Get only NEW data
new_data AS (
    SELECT u.*
    FROM data_governance.silver_cost_monitoring.billing_usage u
    CROSS JOIN last_run lr
    WHERE _metadata.file_modification_time > (SELECT last_ts FROM last_run)
),

-- STEP 3: Transform
transformed AS (
    SELECT
        u.account_id,
        u.workspace_id,
        u.record_id,

        u.usage_date,
        u.sku_name,
        u.cloud,
        u.usage_unit,

        u.user_principal AS user_id,
        u.created_by,
        u.resource_owner,

        u.usage_quantity,
        u.usage_duration_minutes,

        p.effective_price,

        (u.usage_quantity * p.effective_price) AS total_cost,

        u.usage_type,
        u.billing_origin_product,
        u.is_serverless,

        CURRENT_TIMESTAMP() AS load_timestamp

    FROM new_data u
    LEFT JOIN data_governance.gold_cost.dim_sku_pricing p
        ON u.sku_name = p.sku_name
        AND u.cloud = p.cloud
        AND u.usage_unit = p.usage_unit
)

-- STEP 4: INSERT ONLY NEW DATA
INSERT INTO data_governance.gold_cost.fact_cost_usage
SELECT * FROM transformed;